# Notebook 3 – Association Rules

## 1. Introdução

### **Grupo G15 – Teammates**
- **Nuno Miguel Rodrigues da Silva (a25417)**  
- **Hugo Francisco Pereira Azevedo (a25431)**  
- **Tiago André Alves Castro (a25434)**  
- **Gonçalo Edi Pinto Homem de Figueiredo (a26747)**  
- **João Miguel Santos Silva Campos Cunha (a23519)**  

Este notebook corresponde ao **Notebook 3 – Association Rules** do Projeto 02 da unidade curricular de **Inteligência Artificial**.

O objetivo deste trabalho é aplicar técnicas de **Regras de Associação** (*Association Rule Mining*) ao mesmo dataset FIFA utilizado nos notebooks anteriores, de forma a identificar **padrões frequentes** e relações relevantes entre atributos dos jogadores.

Mais concretamente, este notebook inclui:
- Definição do **objetivo de negócio** associado às regras de associação;  
- Descrição do **dataset** utilizado (mesmo dataset dos Notebooks 1 e 2);  
- **Preparação** dos dados para o algoritmo Apriori (discretização e one-hot encoding);  
- Aplicação do algoritmo **Apriori** para extração de itemsets frequentes;  
- Geração e análise de **regras de associação** com base em suporte, confiança e lift;  
- Discussão dos **resultados intermédios e finais**;  
- **Conclusão** com análise dos resultados e lições aprendidas.


## 2. Dataset

### 2.1 Fonte do Dataset
O dataset utilizado neste projeto é o mesmo dos notebooks anteriores e é proveniente do Kaggle:

- **FIFA Players Ratings Dataset**  
- URL: https://www.kaggle.com/datasets/rishidamarla/fifa-players-ratings  

Tal como nos restantes notebooks, é utilizada a versão **`fifa_cleaned.csv`**, previamente tratada.

---

### 2.2 Metadados do Dataset
Serão analisados:
- Número de jogadores (linhas);  
- Número de atributos (colunas);  
- Tipos de dados;  
- Algumas das primeiras linhas do dataset.


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

df = pd.read_csv('fifa_cleaned.csv')
df

,id,name,full_name,birth_date,age,height_cm,weight_kgs,positions,nationality,overall_rating,potential,value_euro,wage_euro,preferred_foot,international_reputation(1-5),weak_foot(1-5),skill_moves(1-5),work_rate,body_type,release_clause_euro,club_team,club_rating,club_position,club_jersey_number,club_join_date,contract_end_year,national_team,national_rating,national_team_position,national_jersey_number,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,freekick_accuracy,long_passing,ball_control,acceleration,sprint_speed,agility,reactions,balance,shot_power,jumping,stamina,strength,long_shots,aggression,interceptions,positioning,vision,penalties,composure,marking,standing_tackle,sliding_tackle,GK_diving,GK_handling,GK_kicking,GK_positioning,GK_reflexes,tags,traits,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,CB,RCB,RB
0,158023,L. Messi,Lionel Andrés Messi Cuccittini,1987-06-24,31,170.18,72.1,"CF,RW,ST",Argentina,94,94,110500000.0,565000.0,Left,5,4,4,Medium/ Low,Messi,226500000.0,FC Barcelona,86.0,RW,10.0,2004-07-01,2021,Argentina,82.0,RF,10.0,86,95,70,92,86,97,93,94,89,96,91,86,93,95,95,85,68,72,66,94,48,22,94,94,75,96,33,28,26,6,11,15,14,8,"#Dribbler,#Distance Shooter,#Crosser,#FK Speci...","Finesse Shot,Long Shot Taker (CPU AI Only),Spe...",89+2,89+2,89+2,93+2,93+2,93+2,93+2,93+2,93+2,93+2,93+2,91+2,85+2,85+2,85+2,91+2,64+2,61+2,61+2,61+2,64+2,59+2,48+2,48+2,48+2,59+2
1,190460,C. Eriksen,Christian Dannemann Eriksen,1992-02-14,27,154.94,76.2,"CAM,RM,CM",Denmark,88,89,69500000.0,205000.0,Right,3,5,4,High/ Medium,Lean,133800000.0,Tottenham Hotspur,83.0,LCM,23.0,2013-08-30,2020,Denmark,78.0,CAM,10.0,88,81,52,91,80,84,86,87,89,91,76,73,80,88,81,84,50,92,58,89,46,56,84,91,67,88,59,57,22,9,14,7,7,6,"#Playmaker ,#Crosser,#FK Specialist,#Complete...","Flair,Long Shot Taker (CPU AI Only),Playmaker ...",79+3,79+3,79+3,85+3,84+3,84+3,84+3,85+3,86+3,86+3,86+3,86+3,85+3,85+3,85+3,86+3,71+3,71+3,71+3,71+3,71+3,66+3,57+3,57+3,57+3,66+3
2,195864,P. Pogba,Paul Pogba,1993-03-15,25,190.50,83.9,"CM,CAM",France,88,91,73000000.0,255000.0,Right,4,4,5,High/ Medium,Normal,144200000.0,Manchester United,82.0,LCM,6.0,2016-08-09,2021,France,84.0,RDM,6.0,80,75,75,86,85,87,85,82,90,90,71,79,76,82,66,90,83,88,87,82,78,64,82,88,82,87,63,67,67,5,6,2,4,3,"#Dribbler,#Playmaker ,#Strength,#Complete Mid...","Flair,Long Passer (CPU AI Only),Long Shot Take...",81+3,81+3,81+3,82+3,83+3,83+3,83+3,82+3,84+3,84+3,84+3,83+3,84+3,84+3,84+3,83+3,76+3,77+3,77+3,77+3,76+3,74+3,72+3,72+3,72+3,74+3
3,198219,L. Insigne,Lorenzo Insigne,1991-06-04,27,162.56,59.0,"LW,ST",Italy,88,88,62000000.0,165000.0,Right,3,4,4,High/ Medium,Normal,105400000.0,Napoli,82.0,LS,24.0,2010-07-01,2022,Italy,83.0,LW,10.0,86,77,56,85,74,90,87,77,78,93,94,86,94,83,93,75,53,75,44,84,34,26,83,87,61,83,51,24,22,8,4,14,9,10,"#Speedster,#Dribbler,#Crosser,#Acrobat","Finesse Shot,Long Shot Taker (CPU AI Only),Spe...",78+3,78+3,78+3,86+3,85+3,85+3,85+3,86+3,86+3,86+3,86+3,86+3,78+3,78+3,78+3,86+3,63+3,58+3,58+3,58+3,63+3,58+3,44+3,44+3,44+3,58+3
4,201024,K. Koulibaly,Kalidou Koulibaly,1991-06-20,27,187.96,88.9,CB,Senegal,88,91,60000000.0,135000.0,Right,3,3,2,High/ High,Normal,106500000.0,Napoli,82.0,LCB,26.0,2014-07-01,2021,NaN,NaN,NaN,NaN,30,22,83,68,14,69,28,28,60,63,70,75,50,82,40,55,81,75,94,15,87,88,24,49,33,80,91,88,87,7,11,7,13,5,"#Tackling ,#Tactician ,#Strength,#Complete Def...",Power Header,53+3,53+3,53+3,53+3,54+3,54+3,54+3,53+3,55+3,55+3,55+3,57+3,61+3,61+3,61+3,57+3,73+3,77+3,77+3,77+3,73+3,76+3,85+3,85+3,85+3,76+3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17949,204322,R. McKenzie,Rory McKenzie,1993-10-07,25,175.26,74.

In [4]:
print('Número de linhas:', df.shape[0])
print('Número de colunas:', df.shape[1])

Número de linhas: 17954
Número de colunas: 92


In [5]:
df.describe(include='all')

,id,name,full_name,birth_date,age,height_cm,weight_kgs,positions,nationality,overall_rating,potential,value_euro,wage_euro,preferred_foot,international_reputation(1-5),weak_foot(1-5),skill_moves(1-5),work_rate,body_type,release_clause_euro,club_team,club_rating,club_position,club_jersey_number,club_join_date,contract_end_year,national_team,national_rating,national_team_position,national_jersey_number,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,freekick_accuracy,long_passing,ball_control,acceleration,sprint_speed,agility,reactions,balance,shot_power,jumping,stamina,strength,long_shots,aggression,interceptions,positioning,vision,penalties,composure,marking,standing_tackle,sliding_tackle,GK_diving,GK_handling,GK_kicking,GK_positioning,GK_reflexes,tags,traits,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,CB,RCB,RB
count,17954.000000,17954,17954,17954,17954.000000,17954.000000,17954.000000,17954,17954,17954.000000,17954.000000,1.769900e+04,17708.000000,17954,17954.000000,17954.000000,17954.000000,17954,17954,1.611700e+04,17940,17940.000000,17940,17940.000000,16018,17593,857,857.000000,857,857.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,17954.000000,1417,8137,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889,15889
unique,NaN,16995,17898,6156,NaN,NaN,NaN,890,160,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,9,10,NaN,687,NaN,29,NaN,1731,24,103,NaN,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81,1594,95,95,95,105,103,103,103,105,102,102,102,98,92,92,92,98,96,98,98,98,96,97,109,109,109,97
top,NaN,J. Rodríguez,Adama Traoré,1992-02-29,NaN,NaN,NaN,CB,England,NaN,NaN,NaN,NaN,Right,NaN,NaN,NaN,Medium/ Medium,Normal,NaN,Valencia CF,NaN,SUB,NaN,2018-07-01,2021,Denmark,NaN,SUB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,#Strength,Speed Dribbler (CPU AI Only),61+2,61+2,61+2,63+2,60+2,60+2,60+2,63+2,61+2,61+2,61+2,61+2,58+2,58+2,58+2,61+2,61+2,62+2,62+2,62+2,61+2,64+2,63+2,63+2,63+2,64+2
freq,NaN,8,3,115,NaN,NaN,NaN,2243,1658,NaN,NaN,NaN,NaN,13781,NaN,NaN,NaN,9690,10393,NaN,33,NaN,7870,NaN,1423,4743,23,NaN,389,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468,372,689,689,689,730,727,727,727,730,727,727,727,809,753,753,753,809,703,637,637,637,703,655,613,613,613,655
mean,215411.087780,NaN,NaN,NaN,25.565445,174.946921,75.301047,NaN,NaN,66.240169,71.430935,2.479280e+06,9902.134628,NaN,1.111674,2.945695,2.361034,NaN,NaN,4.622522e+06,NaN,69.364158,NaN,20.297324,NaN,NaN,NaN,76.341890,NaN,12.456243,49.697672,45.358249,52.148212,58.566058,42.755208,55.278991,47.101147,42.688092,52.667428,58.223906,64.696224,64.803498,63.377465,61.821878,63.870057,55.319873,64.955219,63.133842,65.158850,46.852456,55.816531,46.657959,49.857302,53.406260,48.357302,58.680183,47.162861,47.733040,45.705915,16.784449,16.556199,16.388660,16.541049,16.874067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,29758.387106,NaN,NaN,NaN,4.705708,14.029449,7.083684,NaN,NaN,6.963730,6.131339,5.687014e+06,21995.593750,NaN,0.392168,0.663691,0.763223,NaN,NaN,1.129077e+07,NaN,5.091678,NaN,16.464734,NaN,NaN,NaN,4.786667,NaN,8.098157,18.483310,19.640792,17.521819,14.863178,17.802936,19.091876,18.536621,17.565634,15.

## 3. Objetivo de Negócio

O objetivo deste notebook é **descobrir padrões e relações frequentes** entre características dos jogadores FIFA, recorrendo a **regras de associação**.

Em vez de prever uma classe (como no Notebook 1 – Classificação Automática) ou criar grupos de jogadores (como no Notebook 2 – Clustering), aqui pretendemos **identificar combinações de atributos que ocorrem frequentemente em conjunto**.

Este tipo de análise é útil para:
- Identificar perfis típicos de jogadores com determinadas características (ex.: pé preferido, nacionalidade, overall, potencial);  
- Apoiar decisões de **scouting** ao perceber que tipos de jogadores tendem a partilhar certas características;  
- Apoiar análises estratégicas sobre o mercado de jogadores.
